# Qwen actor + original Laya: 15-minute clean pilot
Select a **GPU runtime**, push the updated branch, then run these cells in order.
The five calendar-read tasks are a small end-to-end pilot, not a full-suite or injection-security evaluation.
Downloads/installations occur before the 900-second trial. The trial includes server startup and warmup, so it may finish with a partial panel. Results are generated from actual execution only.


In [ ]:
import os, subprocess
from pathlib import Path
REPO = Path('/content/permission-that-learn-benchmark')
BRANCH = 'feature/agentdojo-blackbox'
URL = 'https://github.com/AtharvaThorat/permission-that-learn-.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', URL, str(REPO)], check=True)
else:
    if not (REPO / '.git').exists():
        raise RuntimeError('Use a real Git clone; this directory is an extracted archive.')
    branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
    if branch != BRANCH or dirty:
        raise RuntimeError('Existing checkout has another branch or local changes; review it before updating.')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO)
print('Working in', REPO)


## Python 3.12
The supplied Colab runtime used Python 3.13, which this vLLM profile does not support. This installs a separate interpreter without changing the notebook kernel.


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
PYTHON = '/content/py312/bin/python'
if not Path(PYTHON).exists():
    subprocess.run(['uv', 'venv', '--seed', '--python', '3.12', '/content/py312'], check=True)
subprocess.run([PYTHON, '--version'], check=True)


## Prepare or repair
Fresh sessions install packages, run CPU tests, download model assets and verify offline Laya loading. Existing prepared sessions only repair the actor dependency pin (Transformers 4.57.6). This step is outside the benchmark time budget.


In [ ]:
WORK = Path('/content/blackbox-colab')
ready = all(p.exists() for p in [WORK / 'assets.json', WORK / 'gate/bin/python', WORK / 'actor/bin/python'])
command = 'repair-actor' if ready else 'prepare'
print('Setup command:', command)
subprocess.run([PYTHON, 'scripts/colab_blackbox.py', command], check=True)


## Run one bounded five-task benchmark
Qwen3-8B (4-bit actor), original Laya CPU gatekeeper, unchanged reference wall. One server and one gate worker are reused. A tool-call probe and Laya warmup precede the tasks. Logs/results are local to the Colab VM.


In [ ]:
result = subprocess.run([PYTHON, 'scripts/benchmark_colab.py', 'run',
                         '--seconds', '900', '--task-seconds', '150', '--native-raw-logs'])
print('Exit:', result.returncode, '(0: finished; 1: infrastructure errors; 2: setup failure or partial/interrupted run)')


## View and download evidence
Use REPORT.md for presentation. Failed, interrupted and unstarted tasks remain visible. Task utility is not gatekeeper accuracy. Save the ZIP before the runtime is reset.


In [ ]:
import json
from IPython.display import Markdown, display
from google.colab import files
latest = json.loads((WORK / 'latest-benchmark.json').read_text())
result_dir = Path(latest['directory'])
display(Markdown((result_dir / 'REPORT.md').read_text()))
files.download(latest['archive'])
